# Metapaths returned for the five drug/chemical–disease pairs

This notebook uses **all returned paths** for the five pair labels discussed, represented by **six test cases**: Imatinib → asthma, Imatinib → allergic asthma, Fluoxetine → long COVID, Aspirin → colorectal cancer, Acetylsalicylate → colorectal cancer, and Sildenafil → Alzheimer disease.

The Imatinib cases use different disease IDs and remain separate. The saved TEST run (November 7, 2026) returned 500 analyses per case: **3,000 paths in total**. We include paths whether or not they satisfy a test asset. The earlier notebook covered only paths matching passing assets.

A **metapath** is `Node type — predicate — Node type — …`. We use the actual knowledge-graph categories, so acetylsalicylate starts with `SmallMolecule`, while the other five cases start with `Drug`.

**How to read the counts:**
- Each pattern counts once per returned path. Duplicate evidence edges with the same predicate do not increase the count.
- If a hop has multiple predicates, all combinations are enumerated. Thus one ranked path may contribute several metapaths; percentages across patterns can exceed 100%.
- Main tables ignore predicate direction, following the requested type–predicate–type definition. Direction-aware summaries are exported separately. Qualifiers and nested support graphs are not expanded.
- Ranks retain the original order within each query. No score threshold is imposed.
- `asset_matching_path_count` counts paths with that pattern that also satisfy at least one asset for the case. This flag is about expected-node presence, not biological correctness.

Everything runs locally on saved files. Outputs are already executed. To rerun, select `.local/TestHarness/.venv/bin/python` as the kernel. The saved `test_results` folder and `passing_asset_ranks.json` are required.

**Folder guide:** notebook outputs go to `../exports/drug_disease/`. Shared generated data is in `../data/`, and editable reference pathways are in `../references/`. See `../START_HERE.txt` for the reading and execution order.

In [1]:
from pathlib import Path
from collections import defaultdict
from itertools import product
import json
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "test_results/pathfinder-test-full-20261107T192900Z").is_dir())
BASE = ROOT / "pathfiner_aragorn_exp"
DATA = BASE / "data"
REFERENCES = BASE / "references"
OUT = BASE / "exports/drug_disease"
OUT.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)
RUN = ROOT / "test_results/pathfinder-test-full-20261107T192900Z"
CASE_IDS = ["TestCase_0", "TestCase_6", "TestCase_2", "TestCase_3", "TestCase_11", "TestCase_16"]
passing_assets = json.loads((DATA / "passing_asset_ranks.json").read_text())
matching_assets = defaultdict(set)
for asset in passing_assets:
    for hit in asset["matching_paths"]:
        for match in hit["matches"]:
            matching_assets[(asset["response_file"], match["graph_id"])].add(asset["asset_id"])


## Reconstruct the returned paths

The helper checks that each bound graph forms a single chain connecting the query endpoints. It orders nodes by connectivity and retains all distinct predicate choices between adjacent nodes. Missing categories, branching graphs, or ambiguous endpoints cause an assertion rather than a guessed pattern.

In [2]:
def binding_ids(binding):
    return binding.get("ids", []) if isinstance(binding, dict) else [b["id"] for b in binding]


def short(value):
    return value.removeprefix("biolink:")


def node_type(node):
    categories = sorted(set(node.get("categories") or []))
    assert categories, "Node has no category"
    labels = [short(x) for x in categories]
    return labels[0] if len(labels) == 1 else "[" + " | ".join(labels) + "]"


def ordered_chain(graph, kg, source, target):
    adjacency = defaultdict(set)
    edge_pairs = defaultdict(list)
    for eid in graph["edges"]:
        edge = kg["edges"][eid]
        s, t = edge["subject"], edge["object"]
        assert s != t, "Self-loop in a bound path"
        adjacency[s].add(t)
        adjacency[t].add(s)
        edge_pairs[frozenset((s, t))].append(eid)
    assert source != target and source in adjacency and target in adjacency
    assert {n for n, neighbors in adjacency.items() if len(neighbors) == 1} == {source, target}
    assert all(len(neighbors) == (1 if n in (source, target) else 2)
               for n, neighbors in adjacency.items()), "Path branches"
    sequence = [source]
    while sequence[-1] != target:
        next_nodes = adjacency[sequence[-1]] - set(sequence)
        assert len(next_nodes) == 1, "Ambiguous or broken path"
        sequence.append(next(iter(next_nodes)))
    assert set(sequence) == set(adjacency), "Disconnected edges"
    hops = [sorted(set(edge_pairs[frozenset((s, t))]))
            for s, t in zip(sequence, sequence[1:])]
    return sequence, hops


def metapaths(sequence, hop_edge_ids, kg):
    types = [node_type(kg["nodes"][n]) for n in sequence]
    options = []
    for s, t, edge_ids in zip(sequence, sequence[1:], hop_edge_ids):
        choices = set()
        for eid in edge_ids:
            edge = kg["edges"][eid]
            assert edge.get("predicate"), "Edge has no predicate"
            direction = "forward" if (edge["subject"], edge["object"]) == (s, t) else "reverse"
            choices.add((short(edge["predicate"]), direction))
        options.append(sorted(choices))
    patterns = set()
    for combination in product(*options):
        plain = directed = types[0]
        for (predicate, direction), category in zip(combination, types[1:]):
            plain += f" — {predicate} — {category}"
            directed += (f" —{predicate}→ {category}" if direction == "forward"
                         else f" ←{predicate}— {category}")
        patterns.add((plain, directed))
    return types, patterns


In [3]:
path_records, pattern_records, case_records = [], [], []
for case_id in CASE_IDS:
    entries = json.loads((RUN / case_id / "assets.json").read_text())
    query_dirs = {entry["query_directory"] for entry in entries}
    assert len(query_dirs) == 1
    response_path = RUN / next(iter(query_dirs)) / "response.json"
    response_file = str(response_path.relative_to(ROOT))
    message = json.loads(response_path.read_text())["message"]
    assert len(message["results"]) == 1
    result = message["results"][0]
    kg = message["knowledge_graph"]
    query_paths = message["query_graph"]["paths"]
    assert len(query_paths) == 1
    qpath_id, qpath = next(iter(query_paths.items()))
    source_ids = binding_ids(result["node_bindings"][qpath["subject"]])
    target_ids = binding_ids(result["node_bindings"][qpath["object"]])
    assert len(source_ids) == len(target_ids) == 1
    source, target = source_ids[0], target_ids[0]
    query_label = f'{kg["nodes"][source]["name"]} → {kg["nodes"][target]["name"]}'
    case_records.append({"case_id": case_id, "query": query_label,
                         "source_id": source, "target_id": target,
                         "source_type": node_type(kg["nodes"][source]),
                         "target_type": node_type(kg["nodes"][target]),
                         "tested_assets": len(entries),
                         "passing_assets": sum(a["case_id"] == case_id for a in passing_assets),
                         "returned_analyses": len(result["analyses"])})
    for rank, analysis in enumerate(result["analyses"], start=1):
        assert set(analysis["path_bindings"]) == {qpath_id}
        graph_ids = binding_ids(analysis["path_bindings"][qpath_id])
        assert len(graph_ids) == 1, "This saved run is expected to have one bound path per analysis"
        gid = graph_ids[0]
        sequence, hop_edges = ordered_chain(message["auxiliary_graphs"][gid], kg, source, target)
        types, variants = metapaths(sequence, hop_edges, kg)
        path_key = response_file + "#" + gid
        matched = sorted(matching_assets.get((response_file, gid), set()))
        path_records.append({"path_key": path_key, "case_id": case_id, "query": query_label,
                             "graph_id": gid, "rank": rank, "score": analysis.get("score"),
                             "hop_count": len(hop_edges), "node_type_sequence": " — ".join(types),
                             "matches_asset": bool(matched), "matching_assets": matched,
                             "node_ids": sequence, "node_names": [kg["nodes"][n].get("name", n) for n in sequence],
                             "node_categories": [kg["nodes"][n]["categories"] for n in sequence],
                             "hop_edge_ids": hop_edges, "response_file": response_file})
        for plain, directed in sorted(variants):
            pattern_records.append({"path_key": path_key, "metapath": plain, "directed_metapath": directed})

paths = pd.DataFrame(path_records)
patterns = pd.DataFrame(pattern_records)
cases = pd.DataFrame(case_records)
assert paths.path_key.is_unique and len(paths) == 3000
assert cases.tested_assets.sum() == 32 and cases.passing_assets.sum() == 10
assert paths.groupby("case_id").size().eq(500).all()
expected_matching = {f + "#" + gid for (f, gid) in matching_assets if any('/' + c + '/' in f for c in CASE_IDS)}
assert set(paths.loc[paths.matches_asset, "path_key"]) == expected_matching
print(f"Validated {len(paths):,} returned paths, including {paths.matches_asset.sum()} paths matching at least one asset.")


Validated 3,000 returned paths, including 63 paths matching at least one asset.


## Overview of the six queries

The five broad pair labels include two different Imatinib disease endpoints. `returned_paths` counts all paths; `asset_matching_paths` is the subset that satisfied any expected-node check.

In [4]:
plain = patterns[["path_key", "metapath"]].drop_duplicates()

def summarize_by_case(pattern_table, label):
    detail = pattern_table[["path_key", label]].drop_duplicates().merge(
        paths[["path_key", "case_id", "rank", "matches_asset"]], on="path_key", validate="many_to_one")
    summary = detail.groupby(["case_id", label]).agg(
        path_count=("path_key", "nunique"), asset_matching_path_count=("matches_asset", "sum"),
        best_rank=("rank", "min"), median_rank=("rank", "median")
    ).reset_index()
    totals = paths.groupby("case_id").size().rename("returned_paths")
    summary = summary.merge(totals, on="case_id").merge(cases[["case_id", "query"]], on="case_id")
    summary["percent_of_query_paths"] = (100 * summary.path_count / summary.returned_paths).round(1)
    return summary.sort_values(["case_id", "path_count", label], ascending=[True, False, True]).reset_index(drop=True)

summary = summarize_by_case(plain, "metapath")
directed_summary = summarize_by_case(patterns, "directed_metapath")
node_summary = summarize_by_case(paths[["path_key", "node_type_sequence"]], "node_type_sequence")
overview = cases.merge(paths.groupby("case_id").agg(
    returned_paths=("path_key", "nunique"), asset_matching_paths=("matches_asset", "sum")), on="case_id")
overview = overview.merge(summary.groupby("case_id").size().rename("distinct_metapaths"), on="case_id")
with pd.option_context("display.max_colwidth", None):
    display(overview[["case_id", "query", "source_type", "target_id", "tested_assets", "passing_assets",
                      "returned_paths", "asset_matching_paths", "distinct_metapaths"]])


,case_id,query,source_type,target_id,tested_assets,passing_assets,returned_paths,asset_matching_paths,distinct_metapaths
0,TestCase_0,Imatinib → asthma,Drug,MONDO:0004979,4,3,500,11,365
1,TestCase_6,Imatinib → allergic asthma,Drug,MONDO:0004784,4,2,500,45,563
2,TestCase_2,Fluoxetine → long COVID-19,Drug,MONDO:0100233,6,1,500,2,697
3,TestCase_3,Acetylsalicylic acid → colorectal cancer,Drug,MONDO:0005575,7,2,500,2,561
4,TestCase_11,acetylsalicylate → colorectal cancer,SmallMolecule,MONDO:0005575,7,0,500,0,272
5,TestCase_16,Sildenafil → Alzheimer disease,Drug,MONDO:0004975,4,2,500,3,2515


## Most frequent metapaths for each query

Each table shows the top 10 patterns out of **all 500 paths for that query**. The full CSV includes every pattern. Ties in frequency are ordered alphabetically. A separate node-type table gives a simpler view that ignores edge predicates.

In [5]:
TOP_N = 10
for case_id in CASE_IDS:
    label = cases.set_index("case_id").loc[case_id, "query"]
    display(Markdown(f"### {label} ({case_id})"))
    with pd.option_context("display.max_colwidth", None):
        display(summary.loc[summary.case_id == case_id,
            ["metapath", "path_count", "percent_of_query_paths", "asset_matching_path_count", "best_rank"]].head(TOP_N))
        display(Markdown("**Most frequent node-type sequences**"))
        display(node_summary.loc[node_summary.case_id == case_id,
            ["node_type_sequence", "path_count", "percent_of_query_paths"]].head(5))


### Imatinib → asthma (TestCase_0)

,metapath,path_count,percent_of_query_paths,asset_matching_path_count,best_rank
0,Drug — affects — Gene — physically_interacts_with — Gene — condition_associated_with_gene — Disease,263,52.6,1,4
1,Drug — affects — Gene — physically_interacts_with — Gene — gene_associated_with_condition — Disease,263,52.6,1,4
2,Drug — affects — Gene — associated_with — Disease — coexists_with — Disease,57,11.4,1,2
3,Drug — affects — Gene — associated_with — Disease — affects — Disease,55,11.0,1,2
4,Drug — affects — Gene — associated_with — Disease — causes — Disease,52,10.4,1,2
5,Drug — directly_physically_interacts_with — Gene — physically_interacts_with — Gene — condition_associated_with_gene — Disease,51,10.2,1,7
6,Drug — directly_physically_interacts_with — Gene — physically_interacts_with — Gene — gene_associated_with_condition — Disease,51,10.2,1,7
7,Drug — interacts_with — Gene — physically_interacts_with — Gene — condition_associated_with_gene — Disease,47,9.4,1,7
8,Drug — interacts_with — Gene — physically_interacts_with — Gene — gene_associated_with_condition — Disease,47,9.4,1,7
9,Drug — affects — Gene — actively_involved_in — BiologicalProcess — genetic_association — Disease,38,7.6,2,20


**Most frequent node-type sequences**

,node_type_sequence,path_count,percent_of_query_paths
0,Drug — Gene — Gene — Disease,299,59.8
1,Drug — Gene — Disease — Disease,59,11.8
2,Drug — Gene — BiologicalProcess — Disease,45,9.0
3,Drug — Gene — Disease,39,7.8
4,Drug — Gene — Gene — Disease — Disease,13,2.6


### Imatinib → allergic asthma (TestCase_6)

,metapath,path_count,percent_of_query_paths,asset_matching_path_count,best_rank
4410,Drug — affects — Gene — physically_interacts_with — Gene — affects — SmallMolecule — affects — Disease,248,49.6,1,36
4411,Drug — affects — Gene — physically_interacts_with — Gene — affects — SmallMolecule — causes — Disease,248,49.6,1,36
4412,Drug — affects — Gene — physically_interacts_with — Gene — affects — SmallMolecule — in_clinical_trials_for — Disease,248,49.6,1,36
4413,Drug — interacts_with — Gene — physically_interacts_with — Gene — affects — SmallMolecule — affects — Disease,103,20.6,1,36
4414,Drug — interacts_with — Gene — physically_interacts_with — Gene — affects — SmallMolecule — causes — Disease,103,20.6,1,36
4415,Drug — interacts_with — Gene — physically_interacts_with — Gene — affects — SmallMolecule — in_clinical_trials_for — Disease,103,20.6,1,36
4416,Drug — directly_physically_interacts_with — Gene — physically_interacts_with — Gene — affects — SmallMolecule — affects — Disease,102,20.4,1,36
4417,Drug — directly_physically_interacts_with — Gene — physically_interacts_with — Gene — affects — SmallMolecule — causes — Disease,102,20.4,1,36
4418,Drug — directly_physically_interacts_with — Gene — physically_interacts_with — Gene — affects — SmallMolecule — in_clinical_trials_for — Disease,102,20.4,1,36
4419,Drug — affects — Gene — affects — SmallMolecule — affects — Disease,64,12.8,0,9


... (14 outputs left out)

### Sildenafil → Alzheimer disease (TestCase_16)

,metapath,path_count,percent_of_query_paths,asset_matching_path_count,best_rank
637,Drug — affects — Gene — physically_interacts_with — Gene — physically_interacts_with — Gene — related_to — Disease,91,18.2,0,22
638,Drug — affects — Gene — physically_interacts_with — Gene — physically_interacts_with — Gene — associated_with — Disease,81,16.2,0,22
639,Drug — affects — Gene — physically_interacts_with — Gene — physically_interacts_with — Gene — condition_associated_with_gene — Disease,73,14.6,0,22
640,Drug — affects — Gene — physically_interacts_with — Gene — physically_interacts_with — Gene — gene_associated_with_condition — Disease,73,14.6,0,22
641,Drug — affects — Gene — physically_interacts_with — Gene — physically_interacts_with — Gene — predisposes_to_condition — Disease,72,14.4,0,22
642,Drug — affects — Gene — physically_interacts_with — Gene — condition_associated_with_gene — Disease,61,12.2,0,6
643,Drug — affects — Gene — physically_interacts_with — Gene — gene_associated_with_condition — Disease,61,12.2,0,6
644,Drug — affects — Gene — physically_interacts_with — Protein — physically_interacts_with — Gene — related_to — Disease,36,7.2,0,54
645,Drug — directly_physically_interacts_with — Gene — physically_interacts_with — Gene — physically_interacts_with — Gene — related_to — Disease,34,6.8,0,176
646,Drug — affects — Gene — physically_interacts_with — Protein — physically_interacts_with — Gene — predisposes_to_condition — Disease,33,6.6,0,54


**Most frequent node-type sequences**

,node_type_sequence,path_count,percent_of_query_paths
50,Drug — Gene — Gene — Gene — Disease,100,20.0
51,Drug — Gene — Gene — Disease,73,14.6
52,Drug — Gene — Protein — Gene — Disease,37,7.4
53,Drug — Gene — Pathway — Drug — Disease,30,6.0
54,Drug — Gene — Pathway — Gene — Disease,23,4.6


## Explore one query and the paths behind its top pattern

Change `case_id` to any case in the overview. The table below lists concrete node sequences, original ranks, and whether they matched an asset. To inspect a different pattern, replace `selected_metapath`.

In [6]:
case_id = "TestCase_0"
selected_metapath = summary.loc[summary.case_id == case_id, "metapath"].iloc[0]
print(cases.set_index("case_id").loc[case_id, "query"])
print(selected_metapath)
selected_keys = set(plain.loc[plain.metapath == selected_metapath, "path_key"])
examples = paths[(paths.case_id == case_id) & paths.path_key.isin(selected_keys)].sort_values("rank")
with pd.option_context("display.max_colwidth", None):
    display(examples[["rank", "score", "node_names", "matching_assets", "graph_id"]].head(10))


Imatinib → asthma
Drug — affects — Gene — physically_interacts_with — Gene — condition_associated_with_gene — Disease


,rank,score,node_names,matching_assets,graph_id
3,4,0.983788,"[Imatinib, INPPL1, STAT6, asthma]",[],p_3949
5,6,0.980218,"[Imatinib, TRAF6, IL1RL1, asthma]",[],p_4128
6,7,0.979644,"[Imatinib, PDGFRA, H6PD, asthma]",[],p_6301
7,8,0.977150,"[Imatinib, IRAK4, IL1RL1, asthma]",[],p_8706
10,11,0.976335,"[Imatinib, PAX5, DAXX, asthma]",[],p_5583
11,12,0.974595,"[Imatinib, ID2, PKD2, asthma]",[],p_7208
17,18,0.972065,"[Imatinib, ID2, DCAF4, asthma]",[],p_5357
20,21,0.971219,"[Imatinib, PAX5, NEIL2, asthma]",[],p_4396
24,25,0.967102,"[Imatinib, ID2, ID4, asthma]",[],p_5321
25,26,0.965383,"[Imatinib, IL33, IL1RL1, asthma]",[],p_1577


## Export full tables and path details

The mapping CSV links each metapath to its original bound path. The JSON retains node IDs/categories, edge IDs for every hop, matching asset IDs, and the saved response location. No new service requests or publishing occur.

In [7]:
overview.to_csv(OUT / "drug_disease_query_overview.csv", index=False)
summary.to_csv(OUT / "drug_disease_metapaths_by_query.csv", index=False)
directed_summary.to_csv(OUT / "drug_disease_directed_metapaths_by_query.csv", index=False)
node_summary.to_csv(OUT / "drug_disease_node_types_by_query.csv", index=False)
patterns.merge(paths[["path_key", "case_id", "rank", "matches_asset"]], on="path_key", validate="many_to_one").to_csv(
    OUT / "drug_disease_metapath_path_mapping.csv", index=False)
paths.to_json(DATA / "drug_disease_path_details.json", orient="records", indent=2)
print("Saved five CSVs and drug_disease_path_details.json in", OUT)


Saved five CSVs and drug_disease_path_details.json in /Users/carolinagonzalez/shepherd-new-pathfinder/pathfiner_aragorn_exp/exports/drug_disease


## All 32 drug–disease asset results

This table includes both passing and failed assets. Rank is the first qualifying path in the original response order. This export is also used by the exact-path notebook to check its reproduction of the original asset results.

In [8]:
all_asset_ranks = pd.read_csv(BASE / "exports/ranks/all_asset_rank_checks.csv")
drug_assets = all_asset_ranks[all_asset_ranks.case_id.isin(CASE_IDS)].copy()
drug_assets = drug_assets.drop(columns="query").merge(
    cases[["case_id", "query"]], on="case_id", validate="many_to_one")
drug_assets["first_matching_rank"] = drug_assets.first_matching_rank.astype("Int64")
asset_columns = ["case_id", "asset_id", "query", "expected_nodes", "minimum_required", "status",
                 "first_matching_rank", "matching_path_count", "all_matching_ranks", "response_file"]
assert len(drug_assets) == 32 and (drug_assets.status == "PASSED").sum() == 10
drug_assets[asset_columns].to_csv(BASE / "exports/ranks/drug_disease_all_32_asset_results.csv", index=False)
with pd.option_context("display.max_rows", 40, "display.max_colwidth", 70):
    display(drug_assets[["asset_id", "query", "expected_nodes", "minimum_required", "status", "first_matching_rank"]])

,asset_id,query,expected_nodes,minimum_required,status,first_matching_rank
0,Asset_2,Imatinib → asthma,KIT,1,PASSED,2
1,Asset_3,Imatinib → asthma,Histamine,1,PASSED,402
2,Asset_5,Imatinib → asthma,Mast Cell,1,PASSED,22
3,Asset_1,Imatinib → asthma,KIT; Histamine; Mast Cell,2,FAILED,<NA>
4,Asset_57,acetylsalicylate → colorectal cancer,COX2,1,FAILED,<NA>
5,Asset_61,acetylsalicylate → colorectal cancer,hMSH2,1,FAILED,<NA>
6,Asset_60,acetylsalicylate → colorectal cancer,prostaglandin PGE2,1,FAILED,<NA>
7,Asset_56,acetylsalicylate → colorectal cancer,COX2; NFKB; resolvins; prostaglandin PGE2; hMSH2; hMLH1,2,FAILED,<NA>
8,Asset_59,acetylsalicylate → colorectal cancer,resolvins,1,FAILED,<NA>
9,Asset_58,acetylsalicylate → colorectal cancer,NFKB,1,FAILED,<NA>
